# Sessions & Conversation State

*Notebook 19*

Give your agents conversation memory so they remember prior turns across separate runs.
<br>
<br>

**Topics:**

- Why agents forget between runs — and why that matters

- `SQLiteSession` for automatic conversation history

- In-memory vs persistent (file-based) sessions

- Sharing sessions across multiple agents

- Inspecting and clearing session state

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, SQLiteSession

MODEL = "gpt-5-mini"

print("✅ Ready!")

---

## 🎯 The Problem

Every `Runner.run()` call starts fresh — the agent has no memory of previous turns. This breaks multi-turn experiences like a support bot that must remember a ticket ID, or a personal assistant that should recall your preferences. Sessions solve this.

---

## 🧠 Part 1: Without Sessions — The Forgetting Problem

In [ ]:
agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant. Be concise.",
    model=MODEL
)

# Turn 1

result1 = await Runner.run(agent, input="My name is Alex and I work in finance.")

print(f"Turn 1: {result1.final_output}\n")

# Turn 2 — agent has no memory of Turn 1

result2 = await Runner.run(agent, input="What's my name and what do I do?")

print(f"Turn 2: {result2.final_output}")

The agent doesn't know the name or profession — each `Runner.run()` call is a clean slate.

---

## ✅ Part 2: With Sessions — Automatic Memory

Add `session=session` to `Runner.run()`. The SDK automatically loads conversation history before each run and saves new turns after.

In a real app, the session ID is a stable identifier per conversation — typically a user ID, chat ID, or thread ID — so the same conversation reconnects on the next request.

In [ ]:
# Create a session — in-memory by default
session = SQLiteSession("demo_session")

# Turn 1

result1 = await Runner.run(agent, input="My name is Alex and I work in finance.", session=session)

print(f"Turn 1: {result1.final_output}\n")

# Turn 2 — agent remembers Turn 1

result2 = await Runner.run(agent, input="What's my name and what do I do?", session=session)

print(f"Turn 2: {result2.final_output}\n")

# Turn 3 — agent remembers both previous turns

result3 = await Runner.run(agent, input="Based on my background, what AI tools might be useful for me?", session=session)

print(f"Turn 3: {result3.final_output}")

### 💡 Why This Works

The session stores and replays conversation history across runs. Before each run it loads the full conversation history; after each run it saves the new turn. The agent always sees the complete context. Session growth and context window limits (the maximum amount of text a model can process at once) are covered in **Notebook 20: Persistent Memory**.

---

## 💾 Part 3: Persistent Sessions — Surviving Restarts

In-memory sessions are lost when the kernel restarts. A file-based session writes history to a SQLite file — the same conversation is available next time you open the notebook.

In [ ]:
# File-based session — persists across kernel restarts
persistent_session = SQLiteSession(
    "persistent_demo",
    db_path="demo_sessions.sqlite"
)

result = await Runner.run(agent, input="My favorite programming language is Python and I prefer concise answers.", session=persistent_session)

print(f"Turn 1: {result.final_output}")
print(f"\n💾 History saved to: demo_sessions.sqlite")

#### Reconnect and Prove History Survived

In [ ]:
# Simulate a restart — create a new session object with the same ID and path
reconnected_session = SQLiteSession(
    "persistent_demo",
    db_path="demo_sessions.sqlite"
)

result = await Runner.run(agent, input="What do you remember about my preferences?", session=reconnected_session)

print(f"After reconnect: {result.final_output}")

### 💡 Why This Works

The session ID and `db_path` together identify the conversation store. As long as both match, any new session object connects to the same history — even after a kernel restart. Use in-memory sessions for demos and short tasks; use file-based sessions for anything that needs to survive beyond a single run.

---

## 🤝 Part 4: Shared Sessions Across Agents

A session isn't tied to one agent — it's a conversation store. Multiple agents can share the same session, each seeing the full conversation history.

In [ ]:
# Two specialist agents sharing one session

onboarding_agent = Agent(
    name="OnboardingAgent",
    instructions="You help new users get started. Collect their name, role, and goals.",
    model=MODEL
)

advisor_agent = Agent(
    name="AdvisorAgent",
    instructions="You give personalized advice based on the user's background and goals.",
    model=MODEL
)

shared_session = SQLiteSession("shared_demo")

# --------------------------------------------------------------
print("✅ Shared session ready")

#### Onboarding Collects Context

In [ ]:
# Onboarding agent collects user info

result = await Runner.run(onboarding_agent, input="Hi, I'm Jordan. I'm a marketing manager trying to learn AI tools.", session=shared_session)

print(f"Onboarding: {result.final_output}")

#### Advisor Uses the Context

In [ ]:
# Advisor agent sees the full conversation — knows who Jordan is

result = await Runner.run(advisor_agent, input="What should I focus on first?", session=shared_session)

print(f"Advisor: {result.final_output}")

### 💡 Why This Works

The session ID is shared — both agents read from and write to the same conversation history. This is how handoff-based multi-agent systems maintain context without passing state manually. One tradeoff: all prior turns are visible to every agent using the session, so specialists may be influenced by context that isn't relevant to their role. Share a session when agents are collaborating on the same task and later agents need earlier context (triage → specialist); give specialists their own session when their role is narrow and unrelated turns would distract them.

---

## 🧹 Part 5: Managing Session State

Sessions expose methods to inspect and control their history.

`get_items()` returns a list of dicts, one per turn, with at minimum a `role` (`'user'` or `'assistant'`) and `content` (the message text).

Use `get_items()` to debug what the agent actually saw, and `clear_session()` when a user starts a new chat or you want to drop stale context.

In [ ]:
# Inspect the session history
items = await shared_session.get_items()
print(f"📋 Session history: {len(items)} items stored")

# Inspect stored session items
for item_number, item in enumerate(items, 1):
    role = item.get("role", "unknown")
    content = str(item.get("content", ""))[:80]
    print(f"  {item_number}. [{role}] {content}...")

#### Clear a Session

In [ ]:
# Clear a session to start fresh
await shared_session.clear_session()

items_after = await shared_session.get_items()
print(f"✅ Session cleared: {len(items_after)} items remaining")

---

## 🧹 Cleanup

In [ ]:
# Remove demo SQLite file created in Part 3
Path("demo_sessions.sqlite").unlink(missing_ok=True)

# --------------------------------------------------------------
print("🧹 Cleanup complete")

---

## 💪 Practice Exercises

### Exercise 1: Personal Study Assistant

*Covers: conversation state — persisting context across sessions*

Build a study assistant that remembers what topics a student has covered across multiple sessions.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Personal Study Assistant
# --------------------------------------------------------------
# Objective: Build an assistant that remembers covered topics.

# TODO 1: Create a StudyAssistant agent with instructions to:
#           - Track which topics the student has covered
#           - Suggest what to study next based on history
#           - Answer questions about previously covered material

# TODO 2: Create a persistent SQLiteSession with a file path
#           (so it survives kernel restarts)

# TODO 3: Run three turns:
#           Turn 1: "I just finished studying Python basics and functions"
#           Turn 2: "What should I study next?"
#           Turn 3: "What have I covered so far?"
#           Print each response

# --- Write your code below this line ---

### Exercise 2: Multi-Turn Interview Bot

*Covers: conversation state — building multi-turn memory*

Build an interview bot that asks questions one at a time and remembers all previous answers.

**Hint** — starter instruction template: *"You are conducting an interview. In your first response, ask question 1. After each user answer, ask the next question. After the third answer, summarize what you've learned."*

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Multi-Turn Interview Bot
# --------------------------------------------------------------
# Objective: An interviewer that builds context across turns.

# TODO 1: Create an InterviewBot agent with instructions to:
#           - Ask one focused question at a time
#           - Reference previous answers when asking follow-ups
#           - After 3 questions, summarize what it has learned

# TODO 2: Create an in-memory SQLiteSession

# TODO 3: Run 4 turns simulating an interview:
#           Turn 1: Start the interview (the bot asks its first question)
#           Turn 2: Answer the first question
#           Turn 3: Answer the follow-up question
#           Turn 4: Answer the third question and ask for a summary
#           Print each response

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Sessions store conversation history:**

- `SQLiteSession("id")` — in-memory, lost on restart

- `SQLiteSession("id", db_path="file.sqlite")` — persistent, survives restarts

- Pass `session=session` to `Runner.run()` — that's all it takes
<br>
<br>

**Sessions are shareable:**

- Multiple agents can use the same session — all see the full conversation history

- This is how multi-agent systems maintain context without manual state passing
<br>
<br>

**Sessions are controllable:**

- `get_items()` — inspect the history

- `clear_session()` — deletes all conversation items while keeping the session object
<br>
<br>

**Sessions store history, not long-term facts:**

- Sessions store conversation history — persistent memory across sessions is covered in **Notebook 20: Persistent Memory**

---

## 📍 Next Step

**Notebook 20: Persistent Memory**  

Store facts and preferences that persist across sessions, and learn when and what not to remember.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-19-sessions-and-conversation-state)

---